In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

database_name = "fsdh_artemis"

silver_table_state_vectors = f"{database_name}.silver_orion_state_vectors"

gold_table_mission_metrics = f"{database_name}.gold_orion_mission_metrics"
gold_table_latest_state = f"{database_name}.gold_orion_latest_state"
gold_table_summary = f"{database_name}.gold_orion_summary"

# Artemis II launch time: c
launch_timestamp_utc = "2026-04-01T22:24:00.000"

state_vectors_df = spark.table(silver_table_state_vectors)

mission_metrics_df = (
    state_vectors_df
    .withColumn("launch_timestamp_utc", F.to_timestamp(F.lit(launch_timestamp_utc), "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn(
        "speed_km_s",
        F.sqrt(
            F.pow(F.col("vx_km_s"), 2) +
            F.pow(F.col("vy_km_s"), 2) +
            F.pow(F.col("vz_km_s"), 2)
        )
    )
    .withColumn(
        "distance_from_origin_km",
        F.sqrt(
            F.pow(F.col("x_km"), 2) +
            F.pow(F.col("y_km"), 2) +
            F.pow(F.col("z_km"), 2)
        )
    )
    .withColumn(
        "mission_elapsed_hours",
        (
            F.col("epoch_utc").cast("double") - F.col("launch_timestamp_utc").cast("double")
        ) / 3600.0
    )
)
mission_metrics_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(gold_table_mission_metrics)

In [0]:
state_vectors_df = spark.table(silver_table_state_vectors)

display(
    state_vectors_df
    .orderBy("epoch_utc")
)

In [0]:
mission_metrics_df = (
    state_vectors_df
    .withColumn("launch_timestamp_utc", F.to_timestamp(F.lit(launch_timestamp_utc), "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .withColumn(
        "speed_km_s",
        F.sqrt(
            F.pow(F.col("vx_km_s"), 2) +
            F.pow(F.col("vy_km_s"), 2) +
            F.pow(F.col("vz_km_s"), 2)
        )
    )
    .withColumn(
        "distance_from_origin_km",
        F.sqrt(
            F.pow(F.col("x_km"), 2) +
            F.pow(F.col("y_km"), 2) +
            F.pow(F.col("z_km"), 2)
        )
    )
    .withColumn(
        "mission_elapsed_hours",
        (
            F.col("epoch_utc").cast("double") - F.col("launch_timestamp_utc").cast("double")
        ) / 3600.0
    )
)

display(
    mission_metrics_df
    .orderBy("epoch_utc")
)

In [0]:
mission_metrics_df.write.mode("overwrite").saveAsTable(gold_table_mission_metrics)

print(f"Saved table: {gold_table_mission_metrics}")
display(
    spark.table(gold_table_mission_metrics)
    .orderBy("epoch_utc")
)

In [0]:
latest_state_ranked_df = (
    mission_metrics_df
    .withColumn(
        "row_num_desc",
        F.row_number().over(
            Window.partitionBy("source_file").orderBy(F.col("epoch_utc").desc())
        )
    )
)

latest_state_df = (
    latest_state_ranked_df
    .filter(F.col("row_num_desc") == 1)
    .drop("row_num_desc")
)

display(latest_state_df)

In [0]:
latest_state_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table_latest_state)

print(f"Saved table: {gold_table_latest_state}")
display(spark.table(gold_table_latest_state))

In [0]:
summary_df = (
    mission_metrics_df
    .groupBy("source_file")
    .agg(
        F.min("epoch_utc").alias("mission_start_utc"),
        F.max("epoch_utc").alias("mission_end_utc"),
        F.count("*").alias("state_vector_count"),
        F.min("speed_km_s").alias("min_speed_km_s"),
        F.max("speed_km_s").alias("max_speed_km_s"),
        F.avg("speed_km_s").alias("avg_speed_km_s"),
        F.min("distance_from_origin_km").alias("min_distance_from_origin_km"),
        F.max("distance_from_origin_km").alias("max_distance_from_origin_km")
    )
    .withColumn(
        "mission_duration_hours",
        (
            F.col("mission_end_utc").cast("double") - F.col("mission_start_utc").cast("double")
        ) / 3600.0
    )
)

display(summary_df)

In [0]:
summary_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table_summary)

print(f"Saved table: {gold_table_summary}")
display(spark.table(gold_table_summary))

In [0]:
print("Gold mission metrics:")
display(
    spark.table(gold_table_mission_metrics)
    .select(
        "source_file",
        "epoch_utc",
        "mission_elapsed_hours",
        "speed_km_s",
        "distance_from_origin_km"
    )
    .orderBy("epoch_utc")
)

print("Gold latest state:")
display(spark.table(gold_table_latest_state))

print("Gold summary:")
display(spark.table(gold_table_summary))